# 🏗️ Agentic Project Builder — v4
### An AI that writes, compiles, debugs, and runs complete Python projects autonomously
#
---
#
**Fixes applied vs v3:**
- ✅ **No tool API at all** — `tools`, `tool_choice`, probe logic fully removed.
  The model is prompted in plain text and emits `<tool_call>` tags.
- ✅ **`display_project` skips binary files** — `.pyc`, `.pyo`, and any file that
  fails UTF-8 decoding are silently skipped instead of crashing.

## 1. Install Dependencies

In [18]:
# !pip install openai rich tenacity json-repair -q

## 2. Imports & Client

In [19]:
import os, sys, json, ast, re, shutil, subprocess, py_compile, traceback
from pathlib import Path
from datetime import datetime

try:
    from json_repair import repair_json
    HAS_JSON_REPAIR = True
except ImportError:
    HAS_JSON_REPAIR = False

from openai import OpenAI
from tenacity import retry, stop_after_attempt, wait_exponential
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax
from rich.table import Table

console = Console()

# ── Backend (no tool API used — plain chat only) ───────────────────────────────
client = OpenAI(
    base_url="https://bimetallistic-noncuriously-lawanna.ngrok-free.dev/v1",
    api_key="dummy-key"
)

try:
    _models   = client.models.list()
    model_ids = [m.id for m in _models.data]
    MODEL     = model_ids[0]
    console.print(f"✅ Connected. Models: {model_ids}")
    console.print(f"   Using: [bold cyan]{MODEL}[/bold cyan]")
except Exception as e:
    MODEL = "gpt-oss-20b"
    console.print(f"[yellow]⚠️  Model list failed ({e}). Using MODEL='{MODEL}'[/yellow]")

MAX_TOKENS     = 8192
MAX_TOKENS_CAP = 16384
TEMPERATURE    = 0.1
console.print(f"MODEL={MODEL}  MAX_TOKENS={MAX_TOKENS}  TEMP={TEMPERATURE}")
console.print("[bold green]Mode: pure text prompting — no tool API sent[/bold green]")

✅ Connected. Models: ['gpt-oss-20b']

Using: gpt-oss-20b

MODEL=gpt-oss-20b  MAX_TOKENS=8192  TEMP=0.1

Mode: pure text prompting — no tool API sent

## 3. Workspace Manager

In [20]:
PROJECTS_ROOT = Path("./projects")
PROJECTS_ROOT.mkdir(exist_ok=True)

class Workspace:
    def __init__(self, name: str):
        safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in name)
        self.name = safe
        self.root = PROJECTS_ROOT / safe
        self.root.mkdir(parents=True, exist_ok=True)
        console.print(f"📁 Workspace: [bold]{self.root.resolve()}[/bold]")

    def _safe(self, f: str) -> Path:
        p = (self.root / f).resolve()
        if not str(p).startswith(str(self.root.resolve())):
            raise ValueError(f"Path traversal blocked: {f}")
        return p

    def write(self, f: str, text: str) -> str:
        p = self._safe(f)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(text, encoding="utf-8")
        return str(p)

    def read(self, f: str) -> str:
        p = self._safe(f)
        if not p.exists():
            raise FileNotFoundError(f"{f} not found")
        return p.read_text(encoding="utf-8")

    def delete(self, f: str) -> bool:
        p = self._safe(f)
        if p.exists():
            p.unlink()
            return True
        return False

    def all_files(self) -> list:
        return sorted(
            str(x.relative_to(self.root))
            for x in self.root.rglob("*")
            if x.is_file()
        )

    def source_files(self) -> list:
        """All files excluding compiled bytecode and hidden dirs."""
        skip_exts = {".pyc", ".pyo", ".pyd"}
        skip_dirs = {"__pycache__", ".git", ".mypy_cache"}
        result = []
        for x in self.root.rglob("*"):
            if not x.is_file():
                continue
            if any(part in skip_dirs for part in x.parts):
                continue
            if x.suffix in skip_exts:
                continue
            result.append(str(x.relative_to(self.root)))
        return sorted(result)

    def tree(self) -> str:
        files = self.source_files()
        lines = [f"{self.name}/"]
        for i, f in enumerate(files):
            lines.append(f"  {'└' if i == len(files) - 1 else '├'}── {f}")
        return "\n".join(lines)


_WS: Workspace = None

def _ws() -> Workspace:
    if _WS is None:
        raise RuntimeError("No active workspace")
    return _WS

print("✅ Workspace ready.")

✅ Workspace ready.


## 4. Tool Functions

In [21]:
def plan_project(description: str, project_name: str,
                 files: list = None, entry_point: str = "main.py") -> dict:
    return {
        "status": "plan_accepted",
        "project_name": project_name,
        "entry_point": entry_point,
        "files_planned": files or [],
        "next_step": "Call write_file() for each file, then compile_file(), then run_project().",
    }


def write_file(filename: str, content: str, description: str = "") -> dict:
    try:
        fp = _ws().write(filename, content)
        return {"status": "written", "filename": filename, "full_path": fp,
                "lines": content.count("\n") + 1, "description": description}
    except Exception as e:
        return {"status": "error", "filename": filename, "error": str(e)}


def read_file(filename: str) -> dict:
    try:
        c = _ws().read(filename)
        return {"status": "ok", "filename": filename, "content": c,
                "lines": c.count("\n") + 1}
    except Exception as e:
        return {"status": "error", "filename": filename, "error": str(e)}


def list_files() -> dict:
    return {"status": "ok", "files": _ws().source_files(),
            "count": len(_ws().source_files())}


def show_project_tree() -> dict:
    return {"status": "ok", "tree": _ws().tree()}


def compile_file(filename: str) -> dict:
    if not filename.endswith(".py"):
        return {"status": "skipped", "filename": filename, "reason": "Not a .py file"}
    try:
        content = _ws().read(filename)
    except Exception as e:
        return {"status": "error", "filename": filename, "error": str(e)}
    try:
        ast.parse(content)
    except SyntaxError as e:
        return {"status": "syntax_error", "filename": filename,
                "message": e.msg, "line": e.lineno, "text": e.text, "full_error": str(e)}
    try:
        py_compile.compile(str(_ws()._safe(filename)), doraise=True)
        return {"status": "ok", "filename": filename, "message": "Syntax check passed ✓"}
    except py_compile.PyCompileError as e:
        return {"status": "compile_error", "filename": filename, "full_error": str(e)}


def run_project(entry_point: str, args: list = None,
                timeout: int = 30, stdin_input: str = "") -> dict:
    fp = _ws()._safe(entry_point)
    if not fp.exists():
        return {"status": "error",
                "error": f"'{entry_point}' not found. Files: {_ws().source_files()}"}
    try:
        r = subprocess.run(
            [sys.executable, str(fp)] + (args or []),
            capture_output=True, text=True, timeout=timeout,
            cwd=str(_ws().root), input=stdin_input or None,
        )
        ok = r.returncode == 0
        return {"status": "success" if ok else "runtime_error",
                "entry_point": entry_point, "return_code": r.returncode,
                "stdout": r.stdout[:4000], "stderr": r.stderr[:2000], "success": ok}
    except subprocess.TimeoutExpired:
        return {"status": "timeout", "error": f"Process exceeded {timeout}s"}
    except Exception as e:
        return {"status": "error", "error": str(e)}


def fix_code(filename: str, fixed_content: str, fix_description: str = "") -> dict:
    try:
        _ws().write(filename, fixed_content)
        check = compile_file(filename) if filename.endswith(".py") else {"status": "skipped"}
        return {"status": "fixed", "filename": filename,
                "fix_description": fix_description, "compile_check": check}
    except Exception as e:
        return {"status": "error", "filename": filename, "error": str(e)}


def install_package(package_name: str) -> dict:
    if not all(c.isalnum() or c in "-_.[]>=<" for c in package_name):
        return {"status": "error", "error": f"Unsafe name: {package_name}"}
    try:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", package_name, "-q"],
            capture_output=True, text=True, timeout=60,
        )
        return ({"status": "installed", "package": package_name}
                if r.returncode == 0
                else {"status": "install_failed", "stderr": r.stderr[-300:]})
    except Exception as e:
        return {"status": "error", "error": str(e)}


def delete_file(filename: str) -> dict:
    return {"status": "deleted" if _ws().delete(filename) else "not_found",
            "filename": filename}


TOOL_MAP: dict = {
    "plan_project":      plan_project,
    "write_file":        write_file,
    "read_file":         read_file,
    "list_files":        list_files,
    "show_project_tree": show_project_tree,
    "compile_file":      compile_file,
    "run_project":       run_project,
    "fix_code":          fix_code,
    "install_package":   install_package,
    "delete_file":       delete_file,
}
print(f"✅ {len(TOOL_MAP)} tools: {list(TOOL_MAP)}")

✅ 10 tools: ['plan_project', 'write_file', 'read_file', 'list_files', 'show_project_tree', 'compile_file', 'run_project', 'fix_code', 'install_package', 'delete_file']


## 5. Robust Tool-Call Parser
#
Handles three failure modes seen with GPT-OSS 20B:
- No closing `</tool_call>` tag
- Truncated JSON mid-string
- Valid tag but JSON keys use non-standard field names

In [22]:
_TC_STRICT_RE = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.DOTALL)
_TC_OPEN_RE   = re.compile(r'<tool_call>\s*(\{.*?)(?=<tool_call>|</tool_call>|$)', re.DOTALL)


def _try_parse_json(raw: str) -> dict | None:
    raw = raw.strip()
    if not raw:
        return None
    # 1. As-is
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    # 2. json_repair
    if HAS_JSON_REPAIR:
        try:
            repaired = repair_json(raw)
            if repaired:
                return json.loads(repaired)
        except Exception:
            pass
    # 3. Manual bracket completion
    try:
        depth_curly = depth_square = 0
        in_string = escape_next = False
        last_good = 0
        for i, ch in enumerate(raw):
            if escape_next:
                escape_next = False
                continue
            if ch == '\\' and in_string:
                escape_next = True
                continue
            if ch == '"':
                in_string = not in_string
                if not in_string:
                    last_good = i + 1
                continue
            if in_string:
                continue
            if ch in ('{', '['):
                depth_curly  += ch == '{'
                depth_square += ch == '['
            elif ch in ('}', ']'):
                depth_curly  -= ch == '}'
                depth_square -= ch == ']'
                if depth_curly >= 0 and depth_square >= 0:
                    last_good = i + 1
        candidate = raw[:last_good].rstrip().rstrip(',') if last_good else raw
        candidate += ']' * max(0, depth_square) + '}' * max(0, depth_curly)
        return json.loads(candidate)
    except Exception:
        pass
    return None


class _FakeFn:
    def __init__(self, name, args):
        self.name      = name
        self.arguments = json.dumps(args)


class _FakeTC:
    _n = 0
    def __init__(self, name, args, malformed: bool = False):
        _FakeTC._n  += 1
        self.id       = f"txt_{_FakeTC._n}"
        self.function = _FakeFn(name, args)
        self.malformed = malformed


def parse_text_tool_calls(text: str) -> tuple[list, bool]:
    """Returns (tool_calls, attempted)."""
    out = []
    attempted = '<tool_call>' in text

    # Pass 1: strict (closing tag)
    for m in _TC_STRICT_RE.finditer(text):
        obj = _try_parse_json(m.group(1))
        if obj:
            name = obj.get("name") or obj.get("tool") or obj.get("function")
            args = obj.get("arguments") or obj.get("args") or obj.get("parameters") or {}
            if isinstance(args, str):
                try:
                    args = json.loads(args)
                except Exception:
                    args = {}
            if name and name in TOOL_MAP:
                out.append(_FakeTC(name, args))
    if out:
        return out, attempted

    # Pass 2: loose (no closing tag / truncated)
    for m in _TC_OPEN_RE.finditer(text):
        obj = _try_parse_json(m.group(1))
        if obj:
            name = obj.get("name") or obj.get("tool") or obj.get("function")
            args = obj.get("arguments") or obj.get("args") or obj.get("parameters") or {}
            if isinstance(args, str):
                try:
                    args = json.loads(args)
                except Exception:
                    args = {}
            if name and name in TOOL_MAP:
                console.print(f"  [yellow]🔧 Repaired truncated <tool_call> for '{name}'[/yellow]")
                out.append(_FakeTC(name, args, malformed=True))

    return out, attempted


print(f"✅ Parser ready. json_repair={HAS_JSON_REPAIR}")

✅ Parser ready. json_repair=False


## 6. LLM Caller — plain chat, no tool API

In [23]:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=15))
def call_llm(messages: list, verbose_raw: bool = False, max_tokens_override: int = None):
    """
    Plain chat completion — no `tools`, no `tool_choice`.
    The model is instructed via the system prompt to emit <tool_call> tags.
    """
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=max_tokens_override or MAX_TOKENS,
        temperature=TEMPERATURE,
        # ── intentionally NO tools / tool_choice / parallel_tool_calls ────────
    )
    if verbose_raw:
        ch = resp.choices[0]
        console.print(
            f"  [dim]► finish={ch.finish_reason}  "
            f"content_chars={len(ch.message.content or '')}[/dim]"
        )
    return resp


print("✅ LLM caller ready (plain chat — no tool API).")

✅ LLM caller ready (plain chat — no tool API).


## 7. System Prompt

In [24]:
_NAMES = ", ".join(TOOL_MAP)

SYSTEM_PROMPT = f"""You are an expert autonomous Python software engineering agent.
Build complete, working Python projects from a natural language description.

════════════════════════════════════════════════════════
OUTPUT FORMAT — MANDATORY
════════════════════════════════════════════════════════
Every action you take MUST be expressed as a single <tool_call> block.
Do NOT write prose before or after the tag.
The block must be on its own lines and the JSON must be complete and valid.

<tool_call>
{{"name": "tool_name", "arguments": {{"key": "value"}}}}
</tool_call>

You will receive the tool result immediately. Emit one tool call per response.

════════════════════════════════════════════════════════
AVAILABLE TOOLS: {_NAMES}
════════════════════════════════════════════════════════

plan_project(description, project_name, files=[], entry_point="main.py")
write_file(filename, content, description="")
read_file(filename)
list_files()
compile_file(filename)
run_project(entry_point, args=[], timeout=30, stdin_input="")
fix_code(filename, fixed_content, fix_description="")
install_package(package_name)
delete_file(filename)
show_project_tree()

For write_file and fix_code:
  • Put the ENTIRE file content in "content" / "fixed_content"
  • Escape newlines as \\n and double-quotes as \\"
  • Write complete, runnable code — no # TODO, no pass, no placeholders

════════════════════════════════════════════════════════
WORKFLOW — follow in order
════════════════════════════════════════════════════════
1. plan_project   → declare files and entry point
2. write_file     → write each file completely, one at a time
3. compile_file   → syntax-check after EVERY .py file
4. fix_code       → if syntax_error, read_file then fix_code
5. run_project    → execute; inspect stdout and stderr
6. fix_code       → if runtime_error, fix and run again (max 5 cycles)
7. show_project_tree + one-sentence summary

Rules: stdlib only · no input() · no infinite loops · complete files always

════════════════════════════════════════════════════════
WORKED EXAMPLE (study this carefully)
════════════════════════════════════════════════════════

User: Build a hello world project.

Response 1 — you emit:
<tool_call>
{{"name": "plan_project", "arguments": {{"description": "Hello world project", "project_name": "hello", "files": ["main.py"], "entry_point": "main.py"}}}}
</tool_call>

System returns: {{"status": "plan_accepted", ...}}

Response 2 — you emit:
<tool_call>
{{"name": "write_file", "arguments": {{"filename": "main.py", "content": "def main():\\n    print(\\"Hello, World!\\")\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "description": "Entry point"}}}}
</tool_call>

System returns: {{"status": "written", "lines": 5, ...}}

Response 3 — you emit:
<tool_call>
{{"name": "compile_file", "arguments": {{"filename": "main.py"}}}}
</tool_call>

System returns: {{"status": "ok", ...}}

Response 4 — you emit:
<tool_call>
{{"name": "run_project", "arguments": {{"entry_point": "main.py"}}}}
</tool_call>

System returns: {{"status": "success", "stdout": "Hello, World!\\n", ...}}

Response 5 — you emit:
<tool_call>
{{"name": "show_project_tree", "arguments": {{}}}}
</tool_call>

Done. Follow this exact pattern for every project.
"""

console.print("✅ System prompt ready.")

✅ System prompt ready.

## 8. The Project Builder Agent — v4

In [25]:
class ProjectBuilderAgent:
    """Autonomous agent: plan → write → compile → run → debug."""

    def __init__(self, max_iterations: int = 40, verbose: bool = True, verbose_raw: bool = True):
        self.max_iterations = max_iterations
        self.verbose        = verbose
        self.verbose_raw    = verbose_raw
        self.messages: list = []
        self._system_msg: dict = {}
        self._user_msg:   dict = {}
        self.stats = {
            "tool_calls": 0, "files_written": 0,
            "compile_errors_fixed": 0, "runtime_errors_fixed": 0,
            "iterations": 0, "context_resets": 0,
            "repaired_calls": 0, "truncation_retries": 0,
        }

    def _log(self, s):
        if self.verbose:
            console.print(s)

    def _log_tool(self, name: str, args: dict, result: dict):
        if not self.verbose:
            return
        st  = result.get("status", "?")
        ok  = st in ("ok","written","success","installed","fixed",
                     "plan_accepted","deleted","skipped")
        col = "green" if ok else "red"
        icons = {
            "plan_project":"📋","write_file":"✍️ ","read_file":"📖","list_files":"📂",
            "show_project_tree":"🌲","compile_file":"🔨","run_project":"▶️ ",
            "fix_code":"🔧","install_package":"📦","delete_file":"🗑️ ",
        }
        sa = {k: (str(v)[:50]+"…" if isinstance(v, str) and len(str(v)) > 50 else v)
              for k, v in args.items() if k not in ("content", "fixed_content")}
        console.print(
            f"  {icons.get(name,'🔧')} [bold]{name}[/bold]"
            f"({json.dumps(sa)[1:-1]}) → [{col}]{st}[/{col}]"
        )
        if name == "write_file" and st == "written":
            console.print(f"     📄 {args['filename']} ({result.get('lines','?')} lines)")
        elif name == "compile_file" and st not in ("ok","skipped"):
            console.print(
                f"     [red]❌ Line {result.get('line','?')}: "
                f"{result.get('message', result.get('full_error',''))}[/red]"
            )
        elif name == "run_project":
            if result.get("stdout"):
                console.print(Panel(result["stdout"][:800], title="[green]stdout[/green]", expand=False))
            if result.get("stderr") and not result.get("success"):
                console.print(Panel(result["stderr"][:600], title="[red]stderr[/red]", expand=False))
        elif name == "show_project_tree":
            console.print(result.get("tree", ""))
        elif name == "fix_code" and st == "fixed":
            console.print(f"     🩹 {result.get('fix_description','')}")
            cc = result.get("compile_check", {})
            if cc.get("status") not in ("ok","skipped",None):
                console.print(f"     [red]Still erroring after fix: {cc}[/red]")

    def _run_tool(self, tc) -> str:
        name = tc.function.name
        try:
            args = json.loads(tc.function.arguments)
        except json.JSONDecodeError as e:
            return json.dumps({"status": "error", "error": f"Bad JSON args: {e}"})
        self.stats["tool_calls"] += 1
        if getattr(tc, 'malformed', False):
            self.stats["repaired_calls"] += 1
        if name not in TOOL_MAP:
            result = {"status": "error", "error": f"Unknown tool: {name}"}
        else:
            try:
                result = TOOL_MAP[name](**args)
            except TypeError as e:
                result = {"status": "error", "error": f"Wrong args for {name}: {e}"}
            except Exception:
                result = {"status": "error", "error": traceback.format_exc()[-500:]}
        if name == "write_file" and result.get("status") == "written":
            self.stats["files_written"] += 1
        if name == "fix_code" and result.get("status") == "fixed":
            desc = args.get("fix_description", "").lower()
            if "syntax" in desc or "compile" in desc:
                self.stats["compile_errors_fixed"] += 1
            else:
                self.stats["runtime_errors_fixed"] += 1
        self._log_tool(name, args, result)
        return json.dumps(result, indent=2)

    def _make_poke(self, run_result) -> str:
        done_files = _WS.source_files()
        py_files   = [f for f in done_files if f.endswith(".py")]

        if not done_files:
            fn       = "plan_project"
            args_ex  = '{"description": "short description", "project_name": "proj", "files": ["main.py"], "entry_point": "main.py"}'
        elif py_files and run_result is None:
            f        = py_files[0]
            fn       = "compile_file"
            args_ex  = f'{{"filename": "{f}"}}'
        elif run_result and not run_result.get("success"):
            fn       = "fix_code"
            args_ex  = '{"filename": "main.py", "fixed_content": "# full file here\\n", "fix_description": "fix error"}'
        else:
            fn       = "show_project_tree"
            args_ex  = '{}'

        return (
            f"Your last response did not contain a valid <tool_call> block.\n\n"
            f"Emit exactly this now (replace values as needed, keep closing tag):\n\n"
            f"<tool_call>\n"
            f'{{\"name\": \"{fn}\", \"arguments\": {args_ex}}}\n'
            f"</tool_call>\n\n"
            f"Nothing before the opening tag. Nothing after the closing tag."
        )

    def _reset_context(self, run_result, reason: str = ""):
        self.stats["context_resets"] += 1
        self.messages = [
            self._system_msg,
            self._user_msg,
            {"role": "user", "content": self._make_poke(run_result)},
        ]
        self._log(
            f"[bold red]🔄 Context reset #{self.stats['context_resets']}"
            f"{' — ' + reason if reason else ''}[/bold red]"
        )

    def build(self, description: str, project_name: str = None) -> dict:
        global _WS
        if not project_name:
            project_name = (
                "_".join(w for w in description.lower().split() if w.isalpha())[:28]
                or "project"
            )
        project_name = f"{project_name}_{datetime.now().strftime('%H%M%S')}"
        _WS = Workspace(project_name)
        self.stats = dict.fromkeys(self.stats, 0)

        self._system_msg = {"role": "system", "content": SYSTEM_PROMPT}
        self._user_msg   = {"role": "user",   "content": f"Build this project:\n\n{description.strip()}"}
        self.messages    = [self._system_msg, self._user_msg]

        console.rule(f"[bold cyan]🏗️  {project_name}[/bold cyan]")
        self._log(f"[dim]{description.strip()[:200]}[/dim]\n")

        run_result        = None
        final_output      = None
        no_attempt_streak = 0
        token_budget      = MAX_TOKENS

        for it in range(1, self.max_iterations + 1):
            self.stats["iterations"] = it
            self._log(f"\n[bold dim]── Iteration {it}/{self.max_iterations} ──[/bold dim]")

            try:
                response = call_llm(
                    self.messages,
                    verbose_raw=self.verbose_raw,
                    max_tokens_override=token_budget,
                )
            except Exception as e:
                self._log(f"[red]LLM error: {e}[/red]")
                break

            ch     = response.choices[0]
            msg    = ch.message
            finish = ch.finish_reason
            text   = msg.content or ""

            # ── Truncation: retry with higher budget, don't append to history ──
            if finish == "length" and not text.strip().endswith("</tool_call>"):
                new_budget = min(token_budget * 2, MAX_TOKENS_CAP)
                self.stats["truncation_retries"] += 1
                self._log(
                    f"  [yellow]⚠️  finish=length — truncated. "
                    f"Retrying with max_tokens={new_budget}[/yellow]"
                )
                token_budget = new_budget
                continue

            # Always append as plain assistant message (no tool_calls field needed)
            self.messages.append({"role": "assistant", "content": text})

            # ── Parse tool calls from text ─────────────────────────────────────
            tool_calls, attempted = parse_text_tool_calls(text)
            if tool_calls:
                self._log(
                    f"  [bold magenta]{len(tool_calls)} tool call(s)"
                    f" (repaired={sum(1 for t in tool_calls if getattr(t,'malformed',False))})"
                    f"[/bold magenta]"
                )
            elif attempted:
                self._log("  [yellow]⚠️  Model opened <tool_call> but JSON unrecoverable[/yellow]")
            elif text:
                self._log(Panel(text[:400], title="[dim]Agent (no tool calls)[/dim]", expand=False))

            # ── No usable tool calls ───────────────────────────────────────────
            if not tool_calls:
                # Clean finish after successful run
                if finish == "stop" and run_result and run_result.get("success"):
                    final_output = text
                    self._log("[green bold]✅ Project complete![/green bold]")
                    break

                if attempted:
                    # Model tried but JSON was unrecoverable — gentle nudge
                    no_attempt_streak = 0
                    self.messages.append({
                        "role": "user",
                        "content": (
                            "The <tool_call> JSON was malformed or the closing tag was missing. "
                            "Re-emit the same call with valid JSON and the closing </tool_call> tag. "
                            "Keep JSON arguments short (under 200 chars total)."
                        ),
                    })
                else:
                    no_attempt_streak += 1
                    if no_attempt_streak >= 2:
                        self._reset_context(run_result, "no tool-call attempt for 2 iterations")
                        no_attempt_streak = 0
                        token_budget = MAX_TOKENS
                continue

            # ── Execute tool calls ─────────────────────────────────────────────
            no_attempt_streak = 0
            token_budget = MAX_TOKENS

            for tc in tool_calls:
                result_str = self._run_tool(tc)
                # Always inject result as a user message (no native tool role needed)
                self.messages.append({
                    "role": "user",
                    "content": f"Tool result for {tc.function.name}:\n{result_str}",
                })
                if tc.function.name == "run_project":
                    run_result = json.loads(result_str)

        success = bool(run_result and run_result.get("success"))
        self._summary(project_name, _WS, success, run_result)
        return {
            "success": success,
            "project_name": project_name,
            "workspace_path": str(_WS.root.resolve()),
            "final_output": final_output,
            "run_result": run_result,
            "stats": dict(self.stats),
            "files": _WS.source_files(),
        }

    def _summary(self, name, ws, success, run_result):
        t = Table(title=f"Build — {name}", show_header=True)
        t.add_column("Metric", style="bold")
        t.add_column("Value")
        t.add_row("Status",             "[green]✅ SUCCESS[/green]" if success else "[red]❌ FAILED[/red]")
        t.add_row("Iterations",         str(self.stats["iterations"]))
        t.add_row("Tool Calls",         str(self.stats["tool_calls"]))
        t.add_row("Repaired Calls",     str(self.stats["repaired_calls"]))
        t.add_row("Files Written",      str(self.stats["files_written"]))
        t.add_row("Fixes Applied",      str(self.stats["compile_errors_fixed"] + self.stats["runtime_errors_fixed"]))
        t.add_row("Context Resets",     str(self.stats["context_resets"]))
        t.add_row("Truncation Retries", str(self.stats["truncation_retries"]))
        t.add_row("Workspace",          str(ws.root.resolve()))
        if run_result and run_result.get("stdout"):
            t.add_row("Output", run_result["stdout"][:300])
        console.print(t)


console.print("✅ [bold]ProjectBuilderAgent v4[/bold] ready.")

✅ ProjectBuilderAgent v4 ready.

## 9. Helper Functions
#
**`display_project` fix:** uses `source_files()` (excludes `__pycache__`/`.pyc`)
and catches `UnicodeDecodeError` so binary files are skipped gracefully.

In [26]:
def display_project(result: dict):
    ws = Path(result["workspace_path"])
    console.rule(f"[bold]📁 {result['project_name']}[/bold]")
    for f in result.get("files", []):
        p = ws / f
        if not p.exists():
            continue
        # Skip binary file extensions explicitly
        if p.suffix in {".pyc", ".pyo", ".pyd", ".so", ".dll", ".exe"}:
            continue
        ext = p.suffix.lstrip(".")
        lex = {"py": "python", "json": "json", "md": "markdown",
               "html": "html", "txt": "text", "csv": "text"}.get(ext, "text")
        console.print(f"\n[bold cyan]── {f} ──[/bold cyan]")
        try:
            source = p.read_text(encoding="utf-8")
            console.print(Syntax(source, lex, theme="monokai", line_numbers=True))
        except (UnicodeDecodeError, OSError):
            size = p.stat().st_size
            console.print(f"  [dim](binary file, {size} bytes — skipped)[/dim]")


def inspect_project(result: dict):
    ws = Path(result["workspace_path"])
    s  = result["stats"]
    console.rule(f"[bold cyan]{result['project_name']}[/bold cyan]")
    console.print(
        f"  {'✅' if result['success'] else '❌'}  "
        f"iter={s['iterations']}  calls={s['tool_calls']}  "
        f"repaired={s['repaired_calls']}  files={s['files_written']}  "
        f"fixes={s['compile_errors_fixed']+s['runtime_errors_fixed']}  "
        f"resets={s['context_resets']}"
    )
    total = 0
    for f in result.get("files", []):
        p = ws / f
        if not p.exists():
            continue
        if p.suffix == ".py":
            try:
                n = p.read_text(encoding="utf-8").count("\n") + 1
                total += n
                console.print(f"    📄 {f:<35} {n:>4} lines")
            except UnicodeDecodeError:
                pass
        elif p.suffix not in {".pyc", ".pyo"}:
            console.print(f"    📄 {f:<35} {p.stat().st_size:>4} bytes")
    if total:
        console.print(f"    Total Python lines: {total}")
    rr = result.get("run_result")
    if rr and rr.get("stdout"):
        console.print(Panel(rr["stdout"][:600], title="stdout", expand=False))


def build_batch(projects: list, verbose: bool = True) -> list:
    results = []
    for i, p in enumerate(projects, 1):
        console.rule(f"[bold]Batch {i}/{len(projects)}: {p['name']}[/bold]")
        r = ProjectBuilderAgent(verbose=verbose).build(p["desc"], project_name=p["name"])
        results.append(r)
    return results


print("✅ Helpers ready.")

✅ Helpers ready.


---
# 🚀 Build Projects

## Project 1 — 🧮 Scientific Calculator

In [27]:
result1 = ProjectBuilderAgent(verbose=True, verbose_raw=True).build("""
Build a two-file scientific calculator:

calculator.py — Calculator class with:
  basic: add(a,b), subtract(a,b), multiply(a,b), divide(a,b) raises ValueError on zero
  scientific: power(base,exp), sqrt(x), log(x,base=10), sin(x), cos(x), tan(x), factorial(n)
  history: list of (expr_str, result); print_history(), clear_history()
  memory: store(v), recall()->v, clear_memory()

main.py — runs 12 calculations covering all methods, prints each as 'expr = result',
uses memory store/recall, prints full history. stdlib only: math.
""", project_name="calculator")

📁 Workspace: /kaggle/working/projects/calculator_082415

────────────────────────────────────────────── 🏗️  calculator_082415 ───────────────────────────────────────────────

Build a two-file scientific calculator:

calculator.py — Calculator class with:
  basic: add(a,b), subtract(a,b), multiply(a,b), divide(a,b) raises ValueError on zero
  scientific: power(base,exp), sq

── Iteration 1/40 ──

► finish=stop  content_chars=232

...

── Iteration 10/40 ──

► finish=stop  content_chars=55

╭───────────────── Agent (no tool calls) ─────────────────╮
│ Two-file scientific calculator with history and memory. │
╰─────────────────────────────────────────────────────────╯

✅ Project complete!

                     Build — calculator_082415                     
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric             ┃ Value                                      ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Status             │ ✅ SUCCESS                                 │
│ Iterations         │ 10                                         │
│ Tool Calls         │ 9                                          │
│ Repaired Calls     │ 0                                          │
│ Files Written      │ 2                                          │
│ Fixes Applied      │ 1                                          │
│ Context Resets     │ 0                                          │
│ Truncation Retries │ 0                                          │
│ Workspace          │ /kaggle/working/projects/calculator_082415 │
│ Output             │ 1 + 2 = 3                                  │
│                    │ 5 - 3 = 2                                  │
│                    │ 4 * 7 = 28                                 │
│                    │ 10 / 2 = 5.0                               │
│                    │ 2 ** 3 = 8.0                               │
│                    │ sqrt(16) = 4.0                             │
│                    │ log(100) = 2.0                             │
│                    │ sin(pi/2) = 1.0                            │
│                    │ cos(0) = 1.0                               │
│                    │ tan(pi/4) = 0.9999999999999999             │
│                    │ 5! = 120                                   │
│                    │ Memory stored: 42                          │
│                    │ Memory after clear: None                   │
│                    │                                            │
│                    │ Calculation History:                       │
│                    │ 1 + 2 = 3                                  │
│                    │ 5 - 3 = 2                                  │
│                    │ 4 * 7 = 28                                 │
│                    │ 10 / 2 = 5.0                               │
│                    │ 2 ** 3 = 8.0                               │
│                    │ sqrt(16) = 4.0                             │
│                    │ log_10(                                    │
└────────────────────┴────────────────────────────────────────────┘

In [28]:
display_project(result1)

────────────────────────────────────────────── 📁 calculator_082415 ───────────────────────────────────────────────

── calculator.py ──

   1 import math                                                                                                   
   2                                                                                                               
   3 class Calculator:                                                                                             
   4     def __init__(self):                                                                                       
   5         self._history = []                                                                                    
   6         self._memory = None                                                                                   
   7                                                                                                               
   8     # Basic operations                                                                                        
   9     def add(self, a, b):                                                                                      
  10         result = a + b                                                                                        
  11         self._record(f"{a} + {b}", result)                                                                    
  12         return result                                                                                         
  13                                                                                                               
  14     def subtract(self, a, b):                                                                                 
  15         result = a - b                                                                                        
  16         self._record(f"{a} - {b}", result)                                                                    
  17         return result                                                                                         
  18                                                                                                               
  19     def multiply(self, a, b):                                                                                 
  20         result = a * b                                                                                        
  21         self._record(f"{a} * {b}", result)                                                                    
  22         return result                                                                                         
  23                                                                                                               
  24     def divide(self, a, b):                                                                                   
  25         if b == 0:                                                                                            
  26             raise ValueError("Division by zero")                                                              
  27         result = a / b                                                                                        
  28         self._record(f"{a} / {b}", result)                                                                    
  29         return result                                                                                         
  30                                                                                                               
  31     # Scientific operations                                                                                   
  32     def power(self, base, [38;2;248;248;242

── main.py ──

   1 from calculator import Calculator                                                                             
   2 import math                                                                                                   
   3                                                                                                               
   4 def main():                                                                                                   
   5     calc = Calculator()                                                                                       
   6                                                                                                               
   7     # Basic operations                                                                                        
   8     print(f"1 + 2 = {calc.add(1, 2)}")                                                                        
   9     print(f"5 - 3 = {calc.subtract(5, 3)}")                                                                   
  10     print(f"4 * 7 = {calc.multiply(4, 7)}")                                                                   
  11     print(f"10 / 2 = {calc.divide(10, 2)}")                                                                   
  12                                                                                                               
  13     # Scientific operations                                                                                   
  14     print(f"2 ** 3 = {calc.power(2, 3)}")                                                                     
  15     print(f"sqrt(16) = {calc.sqrt(16)}")                                                                      
  16     print(f"log(100) = {calc.log(100)}")                                                                      
  17     print(f"sin(pi/2) = {calc.sin(math.pi/2)}")                                                               
  18     print(f"cos(0) = {calc.cos(0)}")                                                                          
  19     print(f"tan(pi/4) = {calc.tan(math.pi/4)}")                                                               
  20     print(f"5! = {calc.factorial(5)}")                                                                        
  21                                                                                                               
  22     # Memory usage                                                                                            
  23     calc.store(42)                                                                                            
  24     print(f"Memory stored: {calc.recall()}")                                                                  
  25     calc.clear_memory()                                                                                       
  26     print(f"Memory after clear: {calc.recall()}")                                                             
  27                                                                                                               
  28     # Print history                                                                          

## Project 2 — 📋 Todo App with SQLite

In [29]:
result2 = ProjectBuilderAgent(verbose=True, verbose_raw=True).build("""
Build a SQLite-backed CLI todo app:

models.py — Todo dataclass: id, title, priority (high/medium/low), done (bool), created_at (str)
db.py — Database class: connects to todos.db, creates schema.
  Methods: add_todo(title,priority)->Todo, complete_todo(id)->bool,
  delete_todo(id)->bool, list_todos(filter='all'|'done'|'pending')->list,
  get_stats()->dict(total,done,pending)
main.py — demo: add 5 todos, print table, complete 2, print pending, print stats,
  delete 1, final listing. stdlib only: sqlite3, dataclasses, datetime.
""", project_name="todo_app")

📁 Workspace: /kaggle/working/projects/todo_app_082503

─────────────────────────────────────────────── 🏗️  todo_app_082503 ────────────────────────────────────────────────

Build a SQLite-backed CLI todo app:

models.py — Todo dataclass: id, title, priority (high/medium/low), done (bool), created_at (str)
db.py — Database class: connects to todos.db, creates schema.
  Me

── Iteration 1/40 ──

► finish=stop  content_chars=195

...

── Iteration 10/40 ──

► finish=stop  content_chars=91

╭─────────────────────────────────── Agent (no tool calls) ────────────────────────────────────╮
│ ✅ Project built: a SQLite-backed CLI todo app with models, database layer, and demo script. │
╰──────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Project complete!

                                    Build — todo_app_082503                                    
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric             ┃ Value                                                                  ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Status             │ ✅ SUCCESS                                                             │
│ Iterations         │ 10                                                                     │
│ Tool Calls         │ 9                                                                      │
│ Repaired Calls     │ 0                                                                      │
│ Files Written      │ 3                                                                      │
│ Fixes Applied      │ 0                                                                      │
│ Context Resets     │ 0                                                                      │
│ Truncation Retries │ 0                                                                      │
│ Workspace          │ /kaggle/working/projects/todo_app_082503                               │
│ Output             │                                                                        │
│                    │ All todos after addition:                                              │
│                    │ ID    Title                Priority Done  Created At                   │
│                    │ ---------------------------------------------------------------------- │
│                    │ 1     Buy milk             high           2026-02-18T08:25:43.603760   │
│                    │ 2     Write report         medium         2026-02-18T08:25:43.613640   │
│                    │ 3                                                                      │
└────────────────────┴────────────────────────────────────────────────────────────────────────┘

In [30]:
display_project(result2)

─────────────────────────────────────────────── 📁 todo_app_082503 ────────────────────────────────────────────────

── db.py ──

   1 import sqlite3                                                                                                
   2 from typing import List, Dict                                                                                 
   3 from datetime import datetime                                                                                 
   4 from models import Todo                                                                                       
   5                                                                                                               
   6 class Database:                                                                                               
   7     def __init__(self, db_path: str = "todos.db"):                                                            
   8         self.db_path = db_path                                                                                
   9         self.conn = sqlite3.connect(self.db_path)                                                             
  10         self.conn.row_factory = sqlite3.Row                                                                   
  11         self._create_schema()                                                                                 
  12                                                                                                               
  13     def _create_schema(self):                                                                                 
  14         cursor = self.conn.cursor()                                                                           
  15         cursor.execute(                                                                                       
  16             """                                                                                               
  17             CREATE TABLE IF NOT EXISTS todos (                                                                
  18                 id INTEGER PRIMARY KEY AUTOINCREMENT,                                                         
  19                 title TEXT NOT NULL,                                                                          
  20                 priority TEXT NOT NULL CHECK(priority IN ('high','medium','low')),                            
  21                 done INTEGER NOT NULL DEFAULT 0,                                                              
  22                 created_at TEXT NOT NULL                                                                      
  23             );                                                                                                
  24             """                                                                                               
  25         )                                                                                                     
  26         self.conn.commit()                                                                                    
  27                                                                                                               
  28     def add_todo(self, title: str, priority: str) -> Todo:                                                    
  29         if priority not in ("high", "medium", "low"):                                                         
  30             raise ValueError("Priority must be 'high', 'medium', or 'low'")                                   
  31         created_at = datetime.utcnow().isoformat()                                                            
  32         cursor = self.conn.cursor()                                                                           
  33         cursor.execute(                                                                                       
  34             "INSERT INTO todos (title, priority, done, created_at) VALUES (?, ?, 0, ?

── main.py ──

   1 from db import Database                                                                                       
   2                                                                                                               
   3                                                                                                               
   4 def print_table(todos):                                                                                       
   5     print("{:<5} {:<20} {:<8} {:<5} {:<20}".format("ID", "Title", "Priority", "Done", "Created At"))          
   6     print("-" * 70)                                                                                           
   7     for t in todos:                                                                                           
   8         print("{:<5} {:<20} {:<8} {:<5} {:<20}".format(t.id, t.title, t.priority, "✓" if t.done else "", t.cre
   9                                                                                                               
  10                                                                                                               
  11 def main():                                                                                                   
  12     db = Database()                                                                                           
  13                                                                                                               
  14     # Add 5 todos                                                                                             
  15     titles = ["Buy milk", "Write report", "Call Alice", "Plan trip", "Read book"]                             
  16     priorities = ["high", "medium", "low", "medium", "low"]                                                   
  17     for title, prio in zip(titles, priorities):                                                               
  18         db.add_todo(title, prio)                                                                              
  19                                                                                                               
  20     print("\nAll todos after addition:")                                                                      
  21     print_table(db.list_todos())                                                                              
  22                                                                                                               
  23     # Complete 2 todos (IDs 1 and 3)                                                                          
  24     db.complete_todo(1)                                                                                       
  25     db.complete_todo(3)                                                                                       
  26                                                                                                               
  27     print("\nPending todos after completing 2 items:")                                                        
  28     print_table(db.

── models.py ──

   1 from dataclasses import dataclass                                                                             
   2 from datetime import datetime                                                                                 
   3                                                                                                               
   4 @dataclass                                                                                                    
   5 class Todo:                                                                                                   
   6     id: int                                                                                                   
   7     title: str                                                                                                
   8     priority: str  # 'high', 'medium', 'low'                                                                  
   9     done: bool                                                                                                
  10     created_at: str                                                                                           
  11                                                                                                               

── todos.db ──

(binary file, 12288 bytes — skipped)

## Project 3 — 📊 CSV Data Analyser

In [31]:
result3 = ProjectBuilderAgent(verbose=True, verbose_raw=True).build("""
Build a CSV sales analysis pipeline:

data_generator.py — writes 50-row 'sales.csv':
  columns: date(YYYY-MM-DD last 90 days), product(5 names),
  region(N/S/E/W), units_sold(10-200), unit_price(9.99-99.99), revenue=units*price
analyser.py — CSVAnalyser: summary(), by_product(), by_region(), top_days(n=5), monthly_trend()
report.py — entry point: generates CSV, analyses, prints all sections. stdlib: random, csv, datetime.
""", project_name="csv_analyser")

📁 Workspace: /kaggle/working/projects/csv_analyser_082546

───────────────────────────────────────────── 🏗️  csv_analyser_082546 ──────────────────────────────────────────────

Build a CSV sales analysis pipeline:

data_generator.py — writes 50-row 'sales.csv':
  columns: date(YYYY-MM-DD last 90 days), product(5 names),
  region(N/S/E/W), units_sold(10-200), unit_price(9.99-

── Iteration 1/40 ──

► finish=stop  content_chars=220

...

── Iteration 10/40 ──

► finish=stop  content_chars=66

╭────────────────────── Agent (no tool calls) ───────────────────────╮
│ The CSV sales analysis pipeline is complete and runs successfully. │
╰────────────────────────────────────────────────────────────────────╯

✅ Project complete!

                       Build — csv_analyser_082546                        
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric             ┃ Value                                             ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Status             │ ✅ SUCCESS                                        │
│ Iterations         │ 10                                                │
│ Tool Calls         │ 9                                                 │
│ Repaired Calls     │ 0                                                 │
│ Files Written      │ 3                                                 │
│ Fixes Applied      │ 0                                                 │
│ Context Resets     │ 0                                                 │
│ Truncation Retries │ 0                                                 │
│ Workspace          │ /kaggle/working/projects/csv_analyser_082546      │
│ Output             │                                                   │
│                    │ === Summary ===                                   │
│                    │ total_rows: 50                                    │
│                    │ total_units: 4852                                 │
│                    │ total_revenue: 243801.42                          │
│                    │                                                   │
│                    │ === By Product ===                                │
│                    │ Whatsit: {'units': 749, 'revenue': 33140.4}       │
│                    │ Thingamajig: {'units': 1455, 'revenue': 64478.26} │
│                    │ Widget: {'units': 1229, 'revenue': 75507.78}      │
│                    │ Doohickey: {'units': 201, 'revenue': 8235.48}     │
│                    │ Gadget: {'units': 12                              │
└────────────────────┴───────────────────────────────────────────────────┘

In [32]:
display_project(result3)

───────────────────────────────────────────── 📁 csv_analyser_082546 ──────────────────────────────────────────────

── analyser.py ──

   1 import csv                                                                                                    
   2 from collections import defaultdict                                                                           
   3 from datetime import datetime                                                                                 
   4                                                                                                               
   5 class CSVAnalyser:                                                                                            
   6     def __init__(self, filename='sales.csv'):                                                                 
   7         self.filename = filename                                                                              
   8         self.data = []                                                                                        
   9         self._load()                                                                                          
  10                                                                                                               
  11     def _load(self):                                                                                          
  12         with open(self.filename, newline='') as csvfile:                                                      
  13             reader = csv.DictReader(csvfile)                                                                  
  14             for row in reader:                                                                                
  15                 # Convert numeric fields                                                                      
  16                 row['units_sold'] = int(row['units_sold'])                                                    
  17                 row['unit_price'] = float(row['unit_price'])                                                  
  18                 row['revenue'] = float(row['revenue'])                                                        
  19                 row['date'] = datetime.strptime(row['date'], '%Y-%m-%d')                                      
  20                 self.data.append(row)                                                                         
  21                                                                                                               
  22     def summary(self):                                                                                        
  23         total_units = sum(r['units_sold'] for r in self.data)                                                 
  24         total_revenue = sum(r['revenue'] for r in self.data)                                                  
  25         return {                                                                                              
  26             'total_rows': len(self.data),                                                                     
  27             'total_units'[38;2;24

── data_generator.py ──

   1 import csv                                                                                                    
   2 import random                                                                                                 
   3 from datetime import datetime, timedelta                                                                      
   4                                                                                                               
   5 def generate_sales_csv(filename='sales.csv', rows=50):                                                        
   6     products = ['Widget', 'Gadget', 'Doohickey', 'Thingamajig', 'Whatsit']                                    
   7     regions = ['N', 'S', 'E', 'W']                                                                            
   8     today = datetime.today()                                                                                  
   9     start_date = today - timedelta(days=90)                                                                   
  10     dates = [start_date + timedelta(days=i) for i in range(91)]                                               
  11     with open(filename, 'w', newline='') as csvfile:                                                          
  12         writer = csv.writer(csvfile)                                                                          
  13         writer.writerow(['date','product','region','units_sold','unit_price','revenue'])                      
  14         for _ in range(rows):                                                                                 
  15             date = random.choice(dates).strftime('%Y-%m-%d')                                                  
  16             product = random.choice(products)                                                                 
  17             region = random.choice(regions)                                                                   
  18             units_sold = random.randint(10,200)                                                               
  19             unit_price = round(random.uniform(9.99,99.99),2)                                                  
  20             revenue = round(units_sold * unit_price,2)                                                        
  21             writer.writerow([date,product[38

── report.py ──

   1 from data_generator import generate_sales_csv                                                                 
   2 from analyser import CSVAnalyser                                                                              
   3                                                                                                               
   4 def print_section(title, content):                                                                            
   5     print(f"\n=== {title} ===")                                                                               
   6     if isinstance(content, dict):                                                                             
   7         for k,v in content.items():                                                                           
   8             print(f"{k}: {v}")                                                                                
   9     elif isinstance(content, list):                                                                           
  10         for item in content:                                                                                  
  11             print(item)                                                                                       
  12     else:                                                                                                     
  13         print(content)                                                                                        
  14                                                                                                               
  15 if __name__ == "__main__":                                                                                    
  16     # Generate data                                                                                           
  17     generate_sales_csv()                                                                                      
  18     # Analyse                                                                                                 
  19     analyser = CSVAnalyser()                                                                                  
  20     # Summary                                                                                                 
  21     print_section("Summary", analyser.summary())                                                              
  22     # By Product                                                                                              
  23     print_section("By Product", analyser.by_product())                                                        
  24     # By Region                                                                                               
  25     print_section("By Region", analyser.by_region())                                                          
  26     # Top 5 Days                                                                                              
  27     print_section("Top 5 Days", analyser.top_days(5))                                                         
  28     # Monthly Trend                                                                                           
  29     print_section("Monthly Trend", analyser.monthly_trend())                                                  
  30                                                                                                               

── sales.csv ──

   1 date,product,region,units_sold,unit_price,revenue                                                             
   2 2026-01-30,Whatsit,N,100,82.28,8228.0                                                                         
   3 2026-01-27,Thingamajig,E,159,48.08,7644.72                                                                    
   4 2026-02-17,Thingamajig,E,189,42.97,8121.33                                                                    
   5 2025-12-28,Widget,W,124,17.61,2183.64                                                                         
   6 2026-01-08,Whatsit,S,29,68.11,1975.19                                                                         
   7 2025-12-23,Doohickey,W,11,25.97,285.67                                                                        
   8 2025-12-11,Thingamajig,S,57,38.09,2171.13                                                                     
   9 2025-11-28,Gadget,W,79,47.25,3732.75                                                                          
  10 2026-02-01,Thingamajig,E,96,14.55,1396.8                                                                      
  11 2026-01-17,Gadget,S,22,98.98,2177.56                                                                          
  12 2026-01-29,Thingamajig,E,112,13.73,1537.76                                                                    
  13 2025-12-23,Widget,W,94,68.8,6467.2                                                                            
  14 2026-01-22,Thingamajig,S,165,65.08,10738.2                                                                    
  15 2026-01-23,Doohickey,S,46,78.59,3615.14                                                                       
  16 2026-02-01,Whatsit,E,16,28.58,457.28                                                                          
  17 2025-12-18,Gadget,S,92,15.07,1386.44                                                                          
  18 2026-02-05,Whatsit,E,43,82.43,3544.49                                                                         
  19 2026-01-31,Widget,E,60,41.27,2476.2                                                                           
  20 2026-01-25,Widget,E,102,68.5,6987.0                                                                           
  21 2025-11-22,Thingamajig,W,197,20.48,4034.56                                                                    
  22 2025-12-08,Gadget,E,184,69.78,12839.52                                                                        
  23 2026-01-19,Widget,W,27,90.68,2448.36                                                                          
  24 2025-12-06,Gadget,N,80,18.15,1452.0                                                                           
  25 2025-12-10,Thingamajig,S,71,32.18,2284.78                                                                     
  26 2026-02-08,Gadget,S,137,42.65,5843.05                                                                         
  27 2025-11-24,Gadget,E,94,38.97,3663.18                                                                          
  28 2026-01-13,Whatsit,N,111,13.19,1464.09                                                                        
  29 2026-02-08,Gadget,W,61,82.08,5006.88                                                                          
  30 2025-11-23,Gadget,W,92,66.49,6117.08                                                                          
  31 2025-12-18,Doohickey,W,92,23.9,2198.8                                                                         
  32 2026-02-09,Widget,S,33,44.11,1455.63                                                                          
  33 2025-11-27,Doohickey,E,11,85.52,940.72                                                                        
  34 2025-12-27,Widget,N,144,35.12,5057.28                                                                         
  35 2026-01-02,Whatsit,S,106,30.22,3203.32             

## Project 4 — 🔐 Password Manager

In [33]:
result4 = ProjectBuilderAgent(verbose=True, verbose_raw=True).build("""
Build a file-based encrypted password manager:

crypto.py: encrypt(text,key)->str via XOR+base64, decrypt(text,key)->str,
  hash_master(pw)->str via hashlib.sha256
vault.py — Vault: stores JSON encrypted in 'vault.enc'.
  add_entry, get_entry, list_sites, delete_entry, generate_password(length=16) via secrets
main.py: vault master='demo_key', 4 entries, list sites, retrieve+mask one pw,
  generate 3 passwords, show vault file size. stdlib: json, base64, hashlib, secrets, pathlib.
""", project_name="password_manager")

📁 Workspace: /kaggle/working/projects/password_manager_082629

─────────────────────────────────────────── 🏗️  password_manager_082629 ────────────────────────────────────────────

Build a file-based encrypted password manager:

crypto.py: encrypt(text,key)->str via XOR+base64, decrypt(text,key)->str,
  hash_master(pw)->str via hashlib.sha256
vault.py — Vault: stores JSON encryp

── Iteration 1/40 ──

► finish=stop  content_chars=217

...

── Iteration 10/40 ──

► finish=stop  content_chars=38

╭──────── Agent (no tool calls) ─────────╮
│ The password manager project is ready. │
╰────────────────────────────────────────╯

✅ Project complete!

                     Build — password_manager_082629                     
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric             ┃ Value                                            ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Status             │ ✅ SUCCESS                                       │
│ Iterations         │ 10                                               │
│ Tool Calls         │ 9                                                │
│ Repaired Calls     │ 0                                                │
│ Files Written      │ 3                                                │
│ Fixes Applied      │ 0                                                │
│ Context Resets     │ 0                                                │
│ Truncation Retries │ 0                                                │
│ Workspace          │ /kaggle/working/projects/password_manager_082629 │
│ Output             │ Sites in vault:                                  │
│                    │ - example.com                                    │
│                    │ - testsite.org                                   │
│                    │ - myblog.net                                     │
│                    │ - shop.io                                        │
│                    │                                                  │
│                    │ example.com password: ***********                │
│                    │                                                  │
│                    │ Generated passwords:                             │
│                    │ 1: 7(39z2HlWo41v^fA                              │
│                    │ 2: $TL7RT9Q0Ihu5#5M                              │
│                    │ 3: e=x&Av=(eEX2NfR^                              │
│                    │                                                  │
│                    │ Vault file size: 408 bytes                       │
│                    │                                                  │
└────────────────────┴──────────────────────────────────────────────────┘

In [34]:
display_project(result4)

─────────────────────────────────────────── 📁 password_manager_082629 ────────────────────────────────────────────

── crypto.py ──

   1 import base64                                                                                                 
   2 import hashlib                                                                                                
   3                                                                                                               
   4 def _xor_bytes(data: bytes, key: bytes) -> bytes:                                                             
   5     """Return data XORed with key, repeating key as necessary."""                                             
   6     key_len = len(key)                                                                                        
   7     return bytes(b ^ key[i % key_len] for i, b in enumerate(data))                                            
   8                                                                                                               
   9 def encrypt(text: str, key: str) -> str:                                                                      
  10     """Encrypt a string using XOR with the provided key and encode the result in base64."""                   
  11     data = text.encode("utf-8")                                                                               
  12     key_bytes = key.encode("utf-8")                                                                           
  13     encrypted = _xor_bytes(data, key_bytes)                                                                   
  14     return base64.b64encode(encrypted).decode("utf-8")                                                        
  15                                                                                                               
  16 def decrypt(text: str, key: str) -> str:                                                                      
  17     """Decrypt a base64-encoded string that was encrypted with XOR and the provided key."""                   
  18     encrypted = base64.b64decode(text.encode("utf-8"))                                                        
  19     key_bytes = key.encode("utf-8")                                                                           
  20     decrypted = _xor_bytes(encrypted, key_bytes)                                                              
  21     return decrypted.decode("utf-8")                                                                          
  22                                                                                                               
  23 def hash_master(pw: str) -> str:                                                                              
  24     """Return the SHA-256 hex digest of the master password."""                                               
  25     return hashlib.sha256(pw.encode("utf-8")).hexdigest()                                                     
  26                                                                                                               

── main.py ──

   1 import os                                                                                                     
   2 from pathlib import Path                                                                                      
   3                                                                                                               
   4 from vault import Vault                                                                                       
   5                                                                                                               
   6 # Master key for demo purposes                                                                                
   7 MASTER_KEY = "demo_key"                                                                                       
   8                                                                                                               
   9 # Path to the encrypted vault file                                                                            
  10 VAULT_PATH = Path("vault.enc")                                                                                
  11                                                                                                               
  12 # Ensure the vault file exists                                                                                
  13 if not VAULT_PATH.exists():                                                                                   
  14     VAULT_PATH.touch()                                                                                        
  15                                                                                                               
  16 # Initialize the vault                                                                                        
  17 vault = Vault(VAULT_PATH, MASTER_KEY)                                                                         
  18                                                                                                               
  19 # Add 4 entries                                                                                               
  20 vault.add_entry("example.com", "alice", "password123")                                                        
  21 vault.add_entry("testsite.org", "bob", "qwerty!@#")                                                           
  22 vault.add_entry("myblog.net", "carol", "letmein$%^")                                                          
  23 vault.add_entry("shop.io", "dave", "12345678")                                                                
  24                                                                                                               
  25 # List all sites                                                                                              
  26 print("Sites in vault:")                                                                                      
  27 for site in vault.list_sites():                                                                               
  28     print(f"- {site}")                                                                                        
  29                                                                                                               
  30 # Retrieve and mask one password                                                                              
  31 site_to_show = "example.com"                                                                                  
  32 entry = vault.get_entry(site_to_show)                                                                         
  33 if entry:                                                                                                     
  34     masked = "*" * len(entry["password"])  # simple mask                                                      
  35     print(f"\n{site_to_show} password: {masked}")  

── vault.enc ──

  1 H29NT30OHRgJFQEKcQgKFEZfTRRVS0VZREcYHDoZCxgJAE9Vf0kEFQ0GCE1zYUVZREVPHz4YFg4LFwlNZUtHCQUWHhgwGQFIVlZPZX9LGFVuRU1

── vault.py ──

   1 import json                                                                                                   
   2 import os                                                                                                     
   3 from pathlib import Path                                                                                      
   4 from typing import Dict, Any                                                                                  
   5                                                                                                               
   6 from crypto import encrypt, decrypt, hash_master                                                              
   7                                                                                                               
   8 class Vault:                                                                                                  
   9     """Simple file-based encrypted vault."""                                                                  
  10                                                                                                               
  11     def __init__(self, path: str | Path, master_key: str):                                                    
  12         self.path = Path(path)                                                                                
  13         self.master_key = master_key                                                                          
  14         self._data: Dict[str, Any] = {}                                                                       
  15         self._load()                                                                                          
  16                                                                                                               
  17     def _load(self) -> None:                                                                                  
  18         if not self.path.exists():                                                                            
  19             self._data = {}                                                                                   
  20             return                                                                                            
  21         with self.path.open("rb") as f:                                                                       
  22             raw = f.read()                                                                                    
  23         if not raw:                                                                                           
  24             self._data = {}                                                                                   
  25             return                                                                                            
  26         try:                                                                                                  
  27             decoded = decrypt(raw.decode("utf-8"), self.master_key)                                           
  28             self._data = json.loads(decoded)                                                                  
  29         except Exception:                                                                                     
  30             # If decryption fails, treat as empty vault                                                       
  31             self._data = {}                                                                                   
  32                                                                                                               
  33     def [38;2;166;226;46;48;2;39;

## Project 5 — 🧩 Text Adventure Game

In [35]:
result5 = ProjectBuilderAgent(verbose=True, verbose_raw=True).build("""
Build a snake game using python text adventure game (NO input() — fully scripted demo):

engine.py: Room dataclass(name,description,exits,items), Player dataclass(name,health,inventory,current_room)
  GameEngine: load_world(rooms), move(dir), take(item), use(item), look(), status(), is_won()->bool
world.py: 6 rooms: entrance→hallway→armory(sword)→garden(healing_herb)
                                   ↓locked_room(master_key)→treasure_room
main.py: scripted sequence visiting all rooms, collecting items, unlocking treasure, winning.
  VICTORY message with full inventory printed at end.
""", project_name="adventure_game")

📁 Workspace: /kaggle/working/projects/adventure_game_082709

──────────────────────────────────────────── 🏗️  adventure_game_082709 ─────────────────────────────────────────────

Build a scripted text adventure game (NO input() — fully scripted demo):

engine.py: Room dataclass(name,description,exits,items), Player dataclass(name,health,inventory,current_room)
  GameEngine: lo

── Iteration 1/40 ──

► finish=stop  content_chars=206

...

── Iteration 10/40 ──

► finish=stop  content_chars=238

╭───────────────────────────────────────────── Agent (no tool calls) ─────────────────────────────────────────────╮
│ The project “adventure_game_082709” contains a fully scripted text adventure that loads a world of six rooms,   │
│ moves the player through them, collects items, unlocks the treasure room, and prints a victory message with the │
│ final inventory.                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Project complete!

                         Build — adventure_game_082709                         
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric             ┃ Value                                                  ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Status             │ ✅ SUCCESS                                             │
│ Iterations         │ 10                                                     │
│ Tool Calls         │ 9                                                      │
│ Repaired Calls     │ 0                                                      │
│ Files Written      │ 3                                                      │
│ Fixes Applied      │ 0                                                      │
│ Context Resets     │ 0                                                      │
│ Truncation Retries │ 0                                                      │
│ Workspace          │ /kaggle/working/projects/adventure_game_082709         │
│ Output             │ You stand at the grand entrance of the ancient castle. │
│                    │ There is nothing of interest here.                     │
│                    │ Exits: north.                                          │
│                    │ You move north to hallway.                             │
│                    │ A long corridor with flickering torches.               │
│                    │ There is nothing of interest here.                     │
│                    │ Exits: south, east, north.                             │
│                    │ You move east to armory.                               │
│                    │ Dusty weapons line the walls.                          │
│                    │ You see: sw                                            │
└────────────────────┴────────────────────────────────────────────────────────┘

In [36]:
display_project(result5)

──────────────────────────────────────────── 📁 adventure_game_082709 ─────────────────────────────────────────────

── engine.py ──

   1 from dataclasses import dataclass, field                                                                      
   2 from typing import Dict, List, Optional                                                                       
   3                                                                                                               
   4 @dataclass                                                                                                    
   5 class Room:                                                                                                   
   6     name: str                                                                                                 
   7     description: str                                                                                          
   8     exits: Dict[str, str]  # direction -> room name                                                           
   9     items: List[str] = field(default_factory=list)                                                            
  10                                                                                                               
  11 @dataclass                                                                                                    
  12 class Player:                                                                                                 
  13     name: str                                                                                                 
  14     health: int                                                                                               
  15     inventory: List[str] = field(default_factory=list)                                                        
  16     current_room: str = ""                                                                                    
  17                                                                                                               
  18 class GameEngine:                                                                                             
  19     def __init__(self):                                                                                       
  20         self.rooms: Dict[str, Room] = {}                                                                      
  21         self.player: Optional[Player] = None                                                                  
  22         self.won: bool = False                                                                                
  23                                                                                                               
  24     def load_world(self, rooms: List[Room]):                                                                  
  25         self.rooms = {room.name: room for room in rooms}                                                      
  26         # set starting room to first room in list                                                             
  27         if rooms:                                                                                             
  28             self.player = Player(name="Adventurer", health=100, current_room=rooms[0].name)                   
  29                                                                                                               
  30     def move(self, direction: str) -> str:                                                                    
  [38;2;

── main.py ──

   1 from engine import GameEngine                                                                                 
   2 from world import rooms                                                                                       
   3                                                                                                               
   4 # Initialize game engine and load world                                                                       
   5 engine = GameEngine()                                                                                         
   6 engine.load_world(rooms)                                                                                      
   7                                                                                                               
   8 # Scripted sequence of actions                                                                                
   9 print(engine.look())                                                                                          
  10 print(engine.move("north"))  # to hallway                                                                     
  11 print(engine.look())                                                                                          
  12 print(engine.move("east"))   # to armory                                                                      
  13 print(engine.look())                                                                                          
  14 print(engine.take("sword"))                                                                                   
  15 print(engine.move("west"))   # back to hallway                                                                
  16 print(engine.move("north"))  # to garden                                                                      
  17 print(engine.look())                                                                                          
  18 print(engine.take("healing_herb"))                                                                            
  19 print(engine.move("east"))   # to locked_room                                                                 
  20 print(engine.look())                                                                                          
  21 print(engine.take("master_key"))                                                                              
  22 print(engine.use("master_key"))                                                                               
  23 print(engine.move("north"))  # to treasure_room                                                               
  24 print(engine.look())                                                                                          
  25                                                                                                               
  26 # Check win condition                                                                                         
  27 if engine.is_won():                                                                                           
  28     print("\nCongratulations! You have won the adventure.")                                                   
  29     print("Your final inventory:")                                                                            
  30     print(engine.player.inventory)                                                                            
  31 else:                                                                                                         
  32     print("\nYou have not yet won the adventure.")                                                            
  33                                                                                                               
  34 [38;2;149;144;119;48;2;39;4

── world.py ──

   1 from engine import Room                                                                                       
   2                                                                                                               
   3 # Define the world rooms                                                                                      
   4 rooms = [                                                                                                     
   5     Room(                                                                                                     
   6         name="entrance",                                                                                      
   7         description="You stand at the grand entrance of the ancient castle.",                                 
   8         exits={"north": "hallway"},                                                                           
   9         items=[]                                                                                              
  10     ),                                                                                                        
  11     Room(                                                                                                     
  12         name="hallway",                                                                                       
  13         description="A long corridor with flickering torches.",                                               
  14         exits={"south": "entrance", "east": "armory", "north": "garden"},                                     
  15         items=[]                                                                                              
  16     ),                                                                                                        
  17     Room(                                                                                                     
  18         name="armory",                                                                                        
  19         description="Dusty weapons line the walls.",                                                          
  20         exits={"west": "hallway"},                                                                            
  21         items=["sword"]                                                                                       
  22     ),                                                                                                        
  23     Room(                                                                                                     
  24         name="garden",                                                                                        
  25         description="A lush garden with a fragrant herb.",                                                    
  26         exits={"south": "hallway", "east": "locked_room"},                                                    
  27         items=["healing_herb"]                                                                                
  28     ),                                                                                                        
  29     Room(                                                                                                     
  30         name="locked_room",                                                                                   
  31         description="A small room with a locked door to the north.",                                          
  32         exits={"west": "garden", "north": "treasure_room"},                                                   
  33         items

## Batch Build

In [37]:
batch_results = build_batch([
    {"name": "fibonacci", "desc": """
     fib.py — fib_recursive(n), fib_iterative(n), fib_memoized(n) via lru_cache,
       fib_generator(limit)->generator. Docstrings and type hints.
     main.py — benchmarks all 4 for n=30 with time.perf_counter,
       prints table(method,result,time_ms), first 15 from generator. stdlib: time, functools.
    """},
    {"name": "log_analyser", "desc": """
     log_generator.py — writes 150 Apache-style lines to access.log.
     parser.py — LogEntry dataclass + parse_log(path)->list via re.
     analyser.py — top_ips(5), status_dist(), top_paths(5), error_rate(), total_bytes().
     main.py — generates, parses, prints full analysis. stdlib: re, collections, dataclasses, random.
    """},
    {"name": "markdown_converter", "desc": """
     parser.py — MarkdownParser.convert(md)->html. Headings, bold, italic, code, lists, links, blockquotes, hr.
     template.py — wrap_html(title,body)->full HTML with embedded CSS.
     main.py — sample markdown covering all features → output.html, prints first 30 lines. stdlib: re.
    """},
], verbose=True)

────────────────────────────────────────────── Batch 1/3: fibonacci ───────────────────────────────────────────────

📁 Workspace: /kaggle/working/projects/fibonacci_082757

─────────────────────────────────────────────── 🏗️  fibonacci_082757 ───────────────────────────────────────────────

fib.py — fib_recursive(n), fib_iterative(n), fib_memoized(n) via lru_cache,
       fib_generator(limit)->generator. Docstrings and type hints.
     main.py — benchmarks all 4 for n=30 with time.perf_c

── Iteration 1/40 ──

...

── Iteration 10/40 ──

► finish=stop  content_chars=78

╭──────────────────────────── Agent (no tool calls) ─────────────────────────────╮
│ Project tree shows main.py, parser.py, template.py, and generated output.html. │
╰────────────────────────────────────────────────────────────────────────────────╯

✅ Project complete!

                                   Build — markdown_converter_083042                                   
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric             ┃ Value                                                                          ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Status             │ ✅ SUCCESS                                                                     │
│ Iterations         │ 10                                                                             │
│ Tool Calls         │ 9                                                                              │
│ Repaired Calls     │ 0                                                                              │
│ Files Written      │ 3                                                                              │
│ Fixes Applied      │ 0                                                                              │
│ Context Resets     │ 0                                                                              │
│ Truncation Retries │ 0                                                                              │
│ Workspace          │ /kaggle/working/projects/markdown_converter_083042                             │
│ Output             │ <!DOCTYPE html>                                                                │
│                    │     <html lang="en">                                                           │
│                    │     <head>                                                                     │
│                    │         <meta charset="UTF-8">                                                 │
│                    │         <meta http-equiv="X-UA-Compatible" content="IE=edge">                  │
│                    │         <meta name="viewport" content="width=device-width, initial-scale=1.0"> │
│                    │         <title>Sample Markdown</title>                                         │
│                    │         <style>                                                                │
│                    │     body {font-family: Ar                                                      │
└────────────────────┴────────────────────────────────────────────────────────────────────────────────┘

## Custom Project — Finance Tracker

In [38]:
custom_result = ProjectBuilderAgent(verbose=True, verbose_raw=True).build("""
Build a personal finance tracker:

models.py — Transaction dataclass: id, date, description, amount(float, negative=expense),
  category(food/transport/entertainment/income/other), tags(list[str])
ledger.py — Ledger backed by ledger.json:
  add, delete, balance, by_category, summary(income/expenses/balance/count), search(kw)
main.py — 10 transactions, table, balance, by_category, summary, search.
  stdlib: json, dataclasses, datetime, pathlib.
""", project_name="finance_tracker")

📁 Workspace: /kaggle/working/projects/finance_tracker_083247

──────────────────────────────────────────── 🏗️  finance_tracker_083247 ────────────────────────────────────────────

Build a personal finance tracker:

models.py — Transaction dataclass: id, date, description, amount(float, negative=expense),
  category(food/transport/entertainment/income/other), tags(list)
led

── Iteration 1/40 ──

► finish=stop  content_chars=204

...

── Iteration 14/40 ──

► finish=stop  content_chars=67

╭─────────────────────── Agent (no tool calls) ───────────────────────╮
│ The personal finance tracker project is ready and fully functional. │
╰─────────────────────────────────────────────────────────────────────╯

✅ Project complete!

                                 Build — finance_tracker_083247                                 
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric             ┃ Value                                                                   ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Status             │ ✅ SUCCESS                                                              │
│ Iterations         │ 14                                                                      │
│ Tool Calls         │ 13                                                                      │
│ Repaired Calls     │ 0                                                                       │
│ Files Written      │ 3                                                                       │
│ Fixes Applied      │ 2                                                                       │
│ Context Resets     │ 0                                                                       │
│ Truncation Retries │ 0                                                                       │
│ Workspace          │ /kaggle/working/projects/finance_tracker_083247                         │
│ Output             │                                                                         │
│                    │ Transactions:                                                           │
│                    │                                                                         │
│                    │ ID   Date       Description            Amount Category     Tags         │
│                    │ ----------------------------------------------------------------------  │
│                    │ 1    2023-01-05 Salary                3000.00 income       paycheck     │
│                    │ 2    2023-01-06 Groceries             -150.75 food         food, weekly │
│                    │ 3    2023                                                               │
└────────────────────┴─────────────────────────────────────────────────────────────────────────┘

In [39]:
display_project(custom_result)

──────────────────────────────────────────── 📁 finance_tracker_083247 ────────────────────────────────────────────

── ledger.json ──

    1 [                                                                                                            
    2   {                                                                                                          
    3     "id": 1,                                                                                                 
    4     "date": "2023-01-05",                                                                                    
    5     "description": "Salary",                                                                                 
    6     "amount": 3000.0,                                                                                        
    7     "category": "income",                                                                                    
    8     "tags": [                                                                                                
    9       "paycheck"                                                                                             
   10     ]                                                                                                        
   11   },                                                                                                         
   12   {                                                                                                          
   13     "id": 2,                                                                                                 
   14     "date": "2023-01-06",                                                                                    
   15     "description": "Groceries",                                                                              
   16     "amount": -150.75,                                                                                       
   17     "category": "food",                                                                                      
   18     "tags": [                                                                                                
   19       "food",                                                                                                
   20       "weekly"                                                                                               
   21     ]                                                                                                        
   22   },                                                                                                         
   23   {                                                                                                          
   24     "id": 3,                                                                                                 
   25     "date": "2023-01-07",                                                                                    
   26     "description": "Bus fare",                                                                               
   27     "amount": -2.5,                                                                                          
   28     "category": "transport",                                                                                 
   29     "tags": [                                                                                                
   30       "public transport"                                                                                     
   31     ]                                                                                                        
   32   },                                                                                                         
   33   {                                                                                                          
   34     "id": 4,                                                                                                 
   35     "date": "2023-01-08",                         

── ledger.py ──

   1 import json                                                                                                   
   2 from pathlib import Path                                                                                      
   3 from datetime import date                                                                                     
   4 from typing import List, Dict                                                                                 
   5                                                                                                               
   6 # Import using absolute path relative to project root                                                         
   7 from models import Transaction                                                                                
   8                                                                                                               
   9 class Ledger:                                                                                                 
  10     def __init__(self, path: Path | str = "ledger.json"):                                                     
  11         self.path = Path(path)                                                                                
  12         if not self.path.exists():                                                                            
  13             self.path.write_text("[]")                                                                        
  14         self._load()                                                                                          
  15                                                                                                               
  16     def _load(self) -> None:                                                                                  
  17         data = json.loads(self.path.read_text())                                                              
  18         self.transactions: List[Transaction] = [                                                              
  19             Transaction(                                                                                      
  20                 id=tx["id"],                                                                                  
  21                 date=date.fromisoformat(tx["date"]),                                                          
  22                 description=tx["description"],                                                                
  23                 amount=tx["amount"],                                                                          
  24                 category=tx["category"],                                                                      
  25                 tags=tx["tags"],                                                                              
  26             ) for tx in data                                                                                  
  27         ]                                                                                                     
  28                                                                                                               
  29     def _save(self) -> None:                                                                                  
  30         data = [                                                                                              
  31             {                                                                                                 
  32                 "id":[

── main.py ──

   1 import json                                                                                                   
   2 from pathlib import Path                                                                                      
   3 from datetime import date                                                                                     
   4 from typing import List                                                                                       
   5                                                                                                               
   6 # Import using absolute path relative to project root                                                         
   7 from models import Transaction                                                                                
   8 from ledger import Ledger                                                                                     
   9                                                                                                               
  10 # Create ledger instance                                                                                      
  11 ledger = Ledger()                                                                                             
  12                                                                                                               
  13 # Add 10 sample transactions                                                                                  
  14 sample_transactions: List[Transaction] = [                                                                    
  15     Transaction(id=1, date=date(2023, 1, 5), description="Salary", amount=3000.0, category="income", tags=["pa
  16     Transaction(id=2, date=date(2023, 1, 6), description="Groceries", amount=-150.75, category="food", tags=["
  17     Transaction(id=3, date=date(2023, 1, 7), description="Bus fare", amount=-2.5, category="transport", tags=[
  18     Transaction(id=4, date=date(2023, 1, 8), description="Movie", amount=-12.0, category="entertainment", tags
  19     Transaction(id=5, date=date(2023, 1, 9), description="Freelance project", amount=800.0, category="income",
  20     Transaction(id=6, date=date(2023, 1, 10), description="Electricity bill", amount=-60.0, category="other", 
  

── models.py ──

   1 from dataclasses import dataclass                                                                             
   2 from datetime import date                                                                                     
   3 from typing import List                                                                                       
   4                                                                                                               
   5 @dataclass                                                                                                    
   6 class Transaction:                                                                                            
   7     id: int                                                                                                   
   8     date: date                                                                                                
   9     description: str                                                                                          
  10     amount: float                                                                                             
  11     category: str                                                                                             
  12     tags: List[str]                                                                                           
  13                                                                                                               

## Session Summary

In [40]:
all_r = [r for r in [result1, result2, result3, result4, result5, custom_result] if r] + batch_results

t = Table(title="📊 Session Summary", show_header=True, header_style="bold magenta")
for col in ("#","Project","Status","Files","Calls","Repaired","Resets","Iters"):
    t.add_column(col)

wins = 0
for i, r in enumerate(all_r, 1):
    s = r["stats"]; ok = r["success"]
    if ok: wins += 1
    t.add_row(
        str(i), r["project_name"][:28],
        "[green]✅[/green]" if ok else "[red]❌[/red]",
        str(s["files_written"]), str(s["tool_calls"]),
        str(s["repaired_calls"]), str(s["context_resets"]),
        str(s["iterations"]),
    )

console.print(t)
console.print(f"\n[bold]Succeeded: {wins}/{len(all_r)}[/bold]")
console.print(f"Projects dir: [cyan]{PROJECTS_ROOT.resolve()}[/cyan]")

                                  📊 Session Summary                                  
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ # ┃ Project                   ┃ Status ┃ Files ┃ Calls ┃ Repaired ┃ Resets ┃ Iters ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 1 │ calculator_082415         │ ✅     │ 2     │ 9     │ 0        │ 0      │ 10    │
│ 2 │ todo_app_082503           │ ✅     │ 3     │ 9     │ 0        │ 0      │ 10    │
│ 3 │ csv_analyser_082546       │ ✅     │ 3     │ 9     │ 0        │ 0      │ 10    │
│ 4 │ password_manager_082629   │ ✅     │ 3     │ 9     │ 0        │ 0      │ 10    │
│ 5 │ adventure_game_082709     │ ✅     │ 3     │ 9     │ 0        │ 0      │ 10    │
│ 6 │ finance_tracker_083247    │ ✅     │ 3     │ 13    │ 0        │ 0      │ 14    │
│ 7 │ fibonacci_082757          │ ✅     │ 2     │ 7     │ 0        │ 0      │ 8     │
│ 8 │ log_analyser_082844       │ ✅     │ 4     │ 11    │ 0        │ 0      │ 12    │
│ 9 │ markdown_converter_083042 │ ✅     │ 3     │ 9     │ 0        │ 0      │ 10    │
└───┴───────────────────────────┴────────┴───────┴───────┴──────────┴────────┴───────┘

Succeeded: 9/9

Projects dir: /kaggle/working/projects

In [41]:
for r in all_r:
    inspect_project(r)
    print()

──────────────────────────────────────────────── calculator_082415 ────────────────────────────────────────────────

✅  iter=10  calls=9  repaired=0  files=2  fixes=1  resets=0

📄 calculator.py                         93 lines

📄 main.py                               34 lines

Total Python lines: 127

...

📄 output.html                         1380 bytes

📄 parser.py                             53 lines

📄 template.py                           35 lines

Total Python lines: 132

╭──────────────────────────────────────────── stdout ────────────────────────────────────────────╮
│ <!DOCTYPE html>                                                                                │
│     <html lang="en">                                                                           │
│     <head>                                                                                     │
│         <meta charset="UTF-8">                                                                 │
│         <meta http-equiv="X-UA-Compatible" content="IE=edge">                                  │
│         <meta name="viewport" content="width=device-width, initial-scale=1.0">                 │
│         <title>Sample Markdown</title>                                                         │
│         <style>                                                                                │
│     body {font-family: Arial, sans-serif; margin: 2rem;}                                       │
│     h1, h2, h3, h4, h5, h6 {color: #333;}                                                      │
│     pre, code {background: #f4f4f4; padding: 0.2rem 0.4rem; border-radius: 3px;}               │
│     blockquote {border-left: 4px solid #ccc; margin: 1rem 0; padding-left: 1rem; color: #555;} │
│     ul, ol {margin-left: 1.5rem;}                                                              │
│     a {color: #0                                                                               │
╰────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Troubleshooting Guide
#
| Symptom | Root Cause | Fix |
|---------|-----------|-----|
| `UnicodeDecodeError` in display | `.pyc` in file list | Fixed in v4: `source_files()` excludes `__pycache__`; `display_project` catches decode errors |
| `<tool_call>` emitted, no closing tag | Model output truncated | v4 loose parser + JSON repair handles this; if persistent raise `MAX_TOKENS_CAP` |
| `context_resets` > 5 | Model not following text-mode format | Switch model or reduce project complexity |
| `truncation_retries` > 0 | `finish_reason=length` hit on large files | Expected — v4 doubles budget automatically |
| Runtime error loop | Logic bug model can't fix | Read `stderr` panel; often wrong import path |
| Project built but `run_result=None` | `run_project` never called | Check iteration log; model may have stopped at `show_project_tree` |